<a href="https://colab.research.google.com/github/ishikabeniwal/CSET343/blob/main/Assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow scikit-image scikit-learn seaborn
from google.colab import drive
drive.mount('/content/drive')



In [ ]:

import os, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

DATA_DIR = "/content/drive/MyDrive/chest_xray"
classes = ["NORMAL", "PNEUMONIA"]

image_paths, labels = [], []

for path in Path(DATA_DIR).rglob("*"):
    if path.is_file() and path.suffix.lower() in [".jpeg", ".jpg", ".png"]:
        if path.parent.name in classes:
            image_paths.append(str(path))
            labels.append(classes.index(path.parent.name))

print("Total images:", len(image_paths))
print("Normal images:", labels.count(0))
print("Pneumonia images:", labels.count(1))

assert len(image_paths) > 0, "Dataset not found. Check DATA_DIR."


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, class_name in enumerate(classes):
    indexes = [j for j, label in enumerate(labels) if label == i]
    for k, idx in enumerate(random.sample(indexes, min(5, len(indexes)))):
        img = Image.open(image_paths[idx]).convert("L")
        axes[i, k].imshow(img, cmap="gray")
        axes[i, k].set_title(class_name)
        axes[i, k].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from skimage.feature import hog, local_binary_pattern
from skimage.feature import graycomatrix, graycoprops

images, features = [], []

for path in image_paths:
    img = Image.open(path).convert("L").resize((128, 128))
    img = np.array(img, dtype=np.uint8)

    # Image normalization for CNN
    images.append(img / 255.0)

    # HOG features
    hog_feature = hog(
        img, orientations=9, pixels_per_cell=(16, 16),
        cells_per_block=(2, 2)
    )

    # LBP features
    lbp = local_binary_pattern(img, 8, 1, method="uniform")
    lbp_hist, _ = np.histogram(
        lbp.ravel(), bins=np.arange(0, 11), density=True
    )

    # GLCM features
    glcm = graycomatrix(
        img, distances=[1], angles=[0],
        levels=256, symmetric=True, normed=True
    )
    glcm_feature = [
        graycoprops(glcm, p)[0, 0]
        for p in ["contrast", "homogeneity", "energy", "correlation"]
    ]

    features.append(np.concatenate([hog_feature, lbp_hist, glcm_feature]))

X = np.array(images, dtype=np.float32)[..., np.newaxis]
F = np.nan_to_num(np.array(features, dtype=np.float32))
y = np.array(labels)

print("CNN image shape:", X.shape)
print("Feature vector shape:", F.shape)


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_temp, F_train, F_temp, y_train, y_temp = train_test_split(
    X, F, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, F_val, F_test, y_val, y_test = train_test_split(
    X_temp, F_temp, y_temp, test_size=0.50,
    random_state=42, stratify=y_temp
)

print("Training:", len(y_train))
print("Validation:", len(y_val))
print("Testing:", len(y_test))


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(128, 128, 1)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

def evaluate(y_true, pred, prob):
    print("Accuracy:", accuracy_score(y_true, pred))
    print("Precision:", precision_score(y_true, pred, zero_division=0))
    print("Recall:", recall_score(y_true, pred, zero_division=0))
    print("F1-score:", f1_score(y_true, pred, zero_division=0))
    print("AUROC:", roc_auc_score(y_true, prob))

evaluate(y_test, y_pred, y_prob)

print(classification_report(
    y_test, y_pred, target_names=classes, zero_division=0
))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")
plt.show()

# Display misclassified images
wrong = np.where(y_test != y_pred)[0]
print("Misclassified images:", len(wrong))

for idx in wrong[:5]:
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    plt.title(f"Actual: {classes[y_test[idx]]}, Predicted: {classes[y_pred[idx]]}")
    plt.axis("off")
    plt.show()


In [ ]:

plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.show()


In [ ]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svm = make_pipeline(
    StandardScaler(),
    SVC(probability=True, random_state=42)
)

params = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf"]
}

grid = GridSearchCV(
    svm, params, cv=3, scoring="accuracy", n_jobs=-1
)
grid.fit(F_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)


In [ ]:

svm_pred = grid.predict(F_test)
svm_prob = grid.predict_proba(F_test)[:, 1]

evaluate(y_test, svm_pred, svm_prob)

results = {
    "CNN": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob)
    ],
    "SVM": [
        accuracy_score(y_test, svm_pred),
        precision_score(y_test, svm_pred, zero_division=0),
        recall_score(y_test, svm_pred, zero_division=0),
        f1_score(y_test, svm_pred, zero_division=0),
        roc_auc_score(y_test, svm_prob)
    ]
}

import pandas as pd

comparison = pd.DataFrame(
    results, index=["Accuracy", "Precision", "Recall", "F1-score", "AUROC"]
).T

print(comparison.round(4))
print("Best performer by accuracy:", comparison["Accuracy"].idxmax())
